# Random Forest Model for Predicting 'seen'

In [20]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler

# Load and preprocess the data
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(['indicator', 'date']).reset_index(drop=True)
data.ffill(inplace=True)

# Feature Engineering
data['days_since_last_seen'] = np.nan
for indicator, group in data.groupby('indicator'):
    seen_dates = group[group['seen'] == 1]['date']
    data.loc[group.index, 'days_since_last_seen'] = group['date'].apply(
        lambda x: (x - seen_dates[seen_dates < x].max()).days if not seen_dates[seen_dates < x].empty else np.nan
    )
data['days_since_last_seen'] = data['days_since_last_seen'].fillna(0).astype(int)

# Rolling and EWM features
data['ewm_seen'] = data.groupby('indicator')['seen'].apply(lambda x: x.ewm(span=3, adjust=False).mean()).reset_index(drop=True)
data['seen_count_last_3'] = data.groupby('indicator')['seen'].apply(lambda x: x.rolling(3, min_periods=1).sum()).reset_index(drop=True)
data['rolling_mean_7'] = data.groupby('indicator')['seen'].transform(lambda x: x.rolling(7, min_periods=1).mean())
data['rolling_mean_14'] = data.groupby('indicator')['seen'].transform(lambda x: x.rolling(14, min_periods=1).mean())
data['rolling_mean_30'] = data.groupby('indicator')['seen'].transform(lambda x: x.rolling(30, min_periods=1).mean())
data['ewm_seen_long'] = data.groupby('indicator')['seen'].transform(lambda x: x.ewm(span=10, adjust=False).mean())
data['total_seen_last_7'] = data.groupby('indicator')['seen'].transform(lambda x: x.rolling(7, min_periods=1).sum())
data['total_days_not_seen_last_7'] = data.groupby('indicator')['seen'].transform(lambda x: 7 - x.rolling(7, min_periods=1).sum())
data['activity_score'] = data.groupby('indicator')['seen'].transform('sum')
data['activity_score_normalized'] = (data['activity_score'] - data['activity_score'].min()) / (data['activity_score'].max() - data['activity_score'].min())

# Time-based features
data['dayofweek'] = data['date'].dt.dayofweek
data['is_weekend'] = data['dayofweek'].isin([5, 6]).astype(int)

# Final feature list
features = [
    'days_since_last_seen', 'ewm_seen', 'seen_count_last_3',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30',
    'ewm_seen_long', 'total_seen_last_7', 'total_days_not_seen_last_7',
    'activity_score_normalized', 'dayofweek', 'is_weekend'
]

# Drop missing rows and scale
data = data.dropna(subset=features + ['seen'])
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

# Train model on 80%
X = data[features]
y = data['seen']
split = int(0.8 * len(data))
X_train, y_train = X.iloc[:split], y.iloc[:split]

model = RandomForestClassifier(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Real-history evaluation (one prediction per indicator)
window = 7
weights = np.exp(np.linspace(-1., 0., window))
weights /= weights.sum()

true_labels = []
predicted_labels = []

for indicator in data['indicator'].unique():
    group = data[data['indicator'] == indicator].sort_values('date')
    if len(group) <= window:
        continue
    X_group = group[features].values
    y_group = group['seen'].values

    X_window = (X_group[-(window + 1):-1] * weights[:, None]).sum(axis=0).reshape(1, -1)
    y_true = y_group[-1]
    X_window_df = pd.DataFrame(X_window, columns=features)
    y_pred = model.predict(X_window_df)[0]

    true_labels.append(y_true)
    predicted_labels.append(y_pred)

# Metrics
report = classification_report(true_labels, predicted_labels)
matrix = confusion_matrix(true_labels, predicted_labels)

print("=== Classification Report ===")
print(report)
print("\n=== Confusion Matrix ===")
print(matrix)



=== Classification Report ===
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       488
           1       0.67      0.68      0.67        84

    accuracy                           0.90       572
   macro avg       0.81      0.81      0.81       572
weighted avg       0.90      0.90      0.90       572


=== Confusion Matrix ===
[[460  28]
 [ 27  57]]


In [22]:
summary_rows = []

for indicator in data['indicator'].unique():
    group = data[data['indicator'] == indicator].sort_values('date')
    if len(group) <= window:
        continue

    X_group = group[features].values
    y_group = group['seen'].values

    # Prepare input
    X_window = (X_group[-(window + 1):-1] * weights[:, None]).sum(axis=0).reshape(1, -1)
    X_window_df = pd.DataFrame(X_window, columns=features)

    y_true = y_group[-1]
    y_pred = model.predict(X_window_df)[0]

    summary_rows.append({
        'Indicator': indicator,
        'Actual Seen': int(y_true),
        'Predicted Seen': int(y_pred)
    })

# Convert to DataFrame
prediction_summary = pd.DataFrame(summary_rows)

# Print or save
prediction_summary.sort_values('Indicator')


,Indicator,Actual Seen,Predicted Seen
0,102.165.16.161,0,0
1,102.68.39.251,0,0
2,103.140.63.154,0,1
3,103.149.249.226,0,0
4,103.216.220.19,0,0
...,...,...,...
567,www.powerlineblog.com,0,0
568,www.sthda.com,0,0
569,www.thetruthaboutguns.com,0,0
570,www.totem.tech,0,0
